# gpt-oss-120b on Colab A100

This notebook brings up `openai/gpt-oss-120b` on a Colab GPU runtime with `vLLM`, exposes an OpenAI-compatible API on `localhost:8000`, and optionally creates a public Cloudflare tunnel.

Notes:
- OpenAI's official guidance says `gpt-oss-120b` is best with `>=60GB VRAM`, and the release notes describe it as fitting within `80GB` memory.
- On `A100 40GB`, this notebook is likely to fail or thrash. `A100 80GB` is the realistic target.
- This notebook is for trying the model directly. Your current `solution3` app still talks to an Ollama-style endpoint, so a small adapter would be the next step if you want to wire this into the app.

Official references:
- https://openai.com/index/introducing-gpt-oss
- https://developers.openai.com/cookbook/articles/gpt-oss/run-vllm
- https://developers.openai.com/cookbook/articles/openai-harmony


In [ ]:
import json
import os
import re
import shutil
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

MODEL_ID = os.environ.get("GPT_OSS_MODEL_ID", "openai/gpt-oss-120b")
HOST = os.environ.get("GPT_OSS_HOST", "0.0.0.0")
PORT = int(os.environ.get("GPT_OSS_PORT", "8000"))
TENSOR_PARALLEL_SIZE = int(os.environ.get("GPT_OSS_TP", "1"))
GPU_MEMORY_UTILIZATION = os.environ.get("GPT_OSS_GPU_MEMORY_UTILIZATION", "0.95")
LOG_DIR = Path(os.environ.get("GPT_OSS_LOG_DIR", "/content/gpt_oss_runtime"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
SERVER_LOG = LOG_DIR / "vllm.log"
SERVER_PID = LOG_DIR / "vllm.pid"
TUNNEL_LOG = LOG_DIR / "cloudflared.log"
BIN_DIR = Path(os.environ.get("GPT_OSS_BIN_DIR", "/content/bin"))
VENV_DIR = Path(os.environ.get("GPT_OSS_VENV_DIR", "/content/gpt_oss_venv"))
VENV_BIN = VENV_DIR / "bin"
VENV_PYTHON = VENV_BIN / "python"
VENV_VLLM = VENV_BIN / "vllm"
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
TRYCLOUDFLARE_PATTERN = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")

def run(command: str) -> subprocess.CompletedProcess[str]:
    print("$", command)
    result = subprocess.run(
        command,
        shell=True,
        check=False,
        executable="/bin/bash",
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr)
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout, stderr=result.stderr)
    return result

def gpu_info() -> list[dict[str, str]]:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            check=True,
            text=True,
            capture_output=True,
        )
    except Exception:
        return []
    rows = []
    for line in result.stdout.splitlines():
        if not line.strip():
            continue
        name, memory = [part.strip() for part in line.split(",", 1)]
        rows.append({"name": name, "memory": memory})
    return rows

def parse_memory_gb(memory_text: str) -> int:
    match = re.search(r"(\d+)", memory_text)
    return int(match.group(1)) // 1024 if match else 0

info = gpu_info()
print(json.dumps(info, ensure_ascii=False, indent=2))
if not info:
    raise RuntimeError("GPU was not detected. Switch the Colab runtime to GPU first.")

max_memory_gb = max(parse_memory_gb(row["memory"]) for row in info)
if max_memory_gb < 60:
    raise RuntimeError(
        f"Detected about {max_memory_gb}GB VRAM. This is below the official rough target for gpt-oss-120b. "
        "Use an A100 80GB runtime or switch to gpt-oss-20b."
    )

print(f"Using model: {MODEL_ID}")
print(f"Detected max VRAM: {max_memory_gb}GB")
print("Set HF_TOKEN in the next cell before starting the server if the model is not already cached.")

In [ ]:
# Set your Hugging Face access token here if needed.
# The model download will fail without it if the runtime does not already have access.

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("HF_TOKEN is not set yet. Fill it here or export it before running the server cell.")
else:
    print("HF_TOKEN is set.")

In [ ]:
# Install vLLM using the current official vLLM recipe for GPT-OSS on A100.
# The older gptoss-specific wheel path is stale because its pinned torch nightly is no longer resolvable.

run("python -m pip install -q --upgrade pip uv")
run(f"rm -rf {VENV_DIR}")
run(f"uv venv --python 3.12 --seed {VENV_DIR}")
run(f"{VENV_PYTHON} -m pip install -q --upgrade pip openai httpx")
run(f"uv pip install --python {VENV_PYTHON} vllm --torch-backend=auto")
print(json.dumps({
    "venv_dir": str(VENV_DIR),
    "python": str(VENV_PYTHON),
    "vllm": str(VENV_VLLM),
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
}, ensure_ascii=False, indent=2))

In [ ]:
def wait_for_server(base_url: str, timeout_seconds: int = 1800) -> None:
    import httpx

    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            response = httpx.get(f"{base_url}/v1/models", timeout=10.0)
            if response.status_code < 500:
                return
        except Exception as exc:
            last_error = exc
        time.sleep(5)
    raise TimeoutError(f"Timed out waiting for vLLM server: {last_error}")

if SERVER_PID.exists():
    try:
        old_pid = int(SERVER_PID.read_text().strip())
        os.kill(old_pid, 15)
        time.sleep(2)
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN before starting the server.")

server_env = os.environ.copy()
server_env["HF_TOKEN"] = HF_TOKEN
server_env["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"

if not VENV_VLLM.exists():
    raise RuntimeError(f"vLLM binary was not found: {VENV_VLLM}")

server_handle = SERVER_LOG.open("wb")
server_process = subprocess.Popen(
    [
        str(VENV_VLLM),
        "serve",
        MODEL_ID,
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--tensor-parallel-size",
        str(TENSOR_PARALLEL_SIZE),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
    ],
    stdout=server_handle,
    stderr=subprocess.STDOUT,
    env=server_env,
    start_new_session=True,
)
SERVER_PID.write_text(str(server_process.pid), encoding="utf-8")

local_base_url = f"http://127.0.0.1:{PORT}"
wait_for_server(local_base_url)
print(json.dumps({
    "model": MODEL_ID,
    "local_base_url": local_base_url,
    "pid": server_process.pid,
    "log": str(SERVER_LOG),
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
}, ensure_ascii=False, indent=2))

In [ ]:
# Quick local smoke test through the OpenAI-compatible API.

from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="EMPTY")
response = client.responses.create(
    model=MODEL_ID,
    instructions="You are a helpful assistant. Use low reasoning effort.",
    input="Summarize in 3 bullet points what MXFP4 quantization is and why it matters.",
)
print(response.output_text)

In [ ]:
# Optional: expose the local vLLM server with a temporary Cloudflare tunnel.

def ensure_cloudflared() -> Path:
    existing = shutil.which("cloudflared")
    if existing:
        return Path(existing)
    BIN_DIR.mkdir(parents=True, exist_ok=True)
    destination = BIN_DIR / "cloudflared"
    urllib.request.urlretrieve(CLOUDFLARED_URL, destination)
    destination.chmod(destination.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["PATH"] = f"{BIN_DIR}:{os.environ.get('PATH', '')}"
    return destination

def wait_for_tunnel_url(log_path: Path, timeout_seconds: int = 120) -> str:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if log_path.exists():
            content = log_path.read_text(encoding="utf-8", errors="ignore")
            match = TRYCLOUDFLARE_PATTERN.search(content)
            if match:
                return match.group(0)
        time.sleep(2)
    raise TimeoutError("Timed out waiting for a trycloudflare URL.")

cloudflared = ensure_cloudflared()
tunnel_handle = TUNNEL_LOG.open("wb")
tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
public_url = wait_for_tunnel_url(TUNNEL_LOG)
print(json.dumps({
    "public_base_url": public_url,
    "cloudflared_pid": tunnel_process.pid,
    "cloudflared_log": str(TUNNEL_LOG),
}, ensure_ascii=False, indent=2))

## Next step for `solution3`

This notebook gives you an OpenAI-compatible local API at `http://127.0.0.1:8000/v1` or a public Cloudflare URL.

Your current `solution3` app still expects an Ollama-style `/api/generate` endpoint. If you want, the next change should be one of these:
- update `app/services/llm.py` so it can talk to an OpenAI-compatible `responses` endpoint
- or add a tiny compatibility proxy in Colab that converts `/api/generate` into `vLLM` calls
